# 07 embed

Question: does `aosabi_core::embed::answer`, the entry of a host that links the engine and the roots
of its app into its own program, answer the JSON body of a GraphQL request with the bytes a request
of the same document answers through the service, and does it answer a body that is no request with
one error and no data.

The side under test is the operation `embed` of `aosabi-devtool execute`, which hands a body to the
service of a loaded behaviour table through the entry. The reference side is the operation
`execute` of the same primitive over the same table, the executor that the served binaries and the
mode `--requests` reach, so the notebook holds the new entry to an answer the conformance suite
judges already. The corpus is the one of `tests/capture/20_corpus/`, every table of the mode
`pylon`, which is the mode whose tables become a service.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain and the gates the notebook rests on: the build of the devtool, the
lints of the workspace and the format of both files of the entry and of the devtool, every cargo
call with `--locked`.

In [ ]:
import json
import os
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import bytediff, paths, run

P = paths(aosabi_root)
OUT = P.out / "07_embed"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTFMT = CARGO_BIN / "rustfmt"
CORPUS = P.tests / "capture" / "20_corpus"

# The gates this notebook rests on, each cargo call with --locked: the build
# of the seam it drives, the lints of the workspace, and the format of the two
# files of the entry and of the devtool, which rustfmt checks because cargo
# fmt takes no --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2024 crates/aosabi-core/src/embed.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-core" / "src" / "embed.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi/src/embed.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi" / "src" / "embed.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "execute", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("corpus         :", P.rel(CORPUS))
print("cargo          :", reported_cargo)
print("seam           :", json.dumps(smoke))
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert smoke == {"subcommand": "execute", "implemented": True, "lands_at": "M4"}, smoke

DIFFS = {}


def batch(requests, name):
    """Send `requests` to `execute` in one process and return its answers by id."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run([DEVTOOL, "execute", "--requests", path], cwd=P.root, timeout_s=timeout_s)
    (OUT / ("%s.answers.ndjson" % name)).write_text(result.out, encoding="utf-8")
    answers = {}
    for line in result.out.splitlines():
        answer = json.loads(line)
        answers[answer["id"]] = answer
    assert len(answers) == len(requests), (len(answers), len(requests))
    return answers


def body_of(case):
    """Returns the JSON body a client sends over HTTP for the request of `case`."""
    body = {"query": case["document"]}
    if case.get("variables") is not None:
        body["variables"] = case["variables"]
    if case.get("operationName") is not None:
        body["operationName"] = case["operationName"]
    return json.dumps(body, ensure_ascii=False)


def failure(message):
    """Returns the response body with the one error `message` and no data."""
    return json.dumps({"errors": [{"message": message}]}, ensure_ascii=False, separators=(",", ":"))

## The corpus through both entries

Every table of the mode `pylon` loads once, and every case of its corpus runs twice in one process,
as an `execute` of its document and as an `embed` of the body a client sends for it. The entry
validates every request, as a server does, so a case that turns validation off has no body that
asks for it and is counted and left out. Where the executor answers a result, the body is the
result text byte for byte. Where the resolver throws a value that is no GraphQL error, the executor
answers `thrown` and the body is the one error with the message of the thrown error, and the fixed
message for a thrown value that is no error.

In [ ]:
CORPORA = sorted(CORPUS.glob("*-pylon.json"))
compared = 0
skipped_unvalidated = 0
thrown_cases = 0
by_corpus = {}
for corpus_path in CORPORA:
    corpus = json.loads(corpus_path.read_text(encoding="utf-8"))
    requests = []
    for index, table in enumerate(corpus["tables"]):
        requests.append({"id": "t%d" % index, "op": "table", "table": table})
    pairs = []
    for index, case in enumerate(corpus["cases"]):
        if case.get("validate", True) is False:
            skipped_unvalidated += 1
            continue
        request = {"id": "x%d" % index, "op": "execute", "table": case["table"], "document": case["document"]}
        for key in ("variables", "operationName"):
            if case.get(key) is not None:
                request[key] = case[key]
        requests.append(request)
        requests.append({"id": "e%d" % index, "op": "embed", "table": case["table"], "body": body_of(case)})
        pairs.append(index)
    answers = batch(requests, corpus_path.stem)
    for index in range(len(corpus["tables"])):
        loaded = answers["t%d" % index]
        assert loaded.get("ok") is True or "errors" in loaded, loaded
    differing = 0
    for index in pairs:
        executed = answers["x%d" % index]
        embedded = answers["e%d" % index]
        assert "fatal" not in executed and "fatal" not in embedded, (executed, embedded)
        if "result" in executed:
            expected = executed["result"]
        else:
            thrown_cases += 1
            thrown = executed["thrown"]
            message = thrown.get("message")
            if thrown.get("name") is None or message is None:
                message = "the request threw a value that is no error"
            expected = failure(message)
        if expected != embedded["body"]:
            differing += 1
            if differing <= 3:
                bytediff(expected.encode(), embedded["body"].encode(), "%s case %d" % (corpus_path.stem, index))
        compared += 1
    by_corpus[corpus_path.stem] = (len(pairs), differing)
    print("%-28s %4d cases, %d differ" % (corpus_path.stem, len(pairs), differing))
DIFFS["corpus"] = sum(d for _, d in by_corpus.values())
print("compared:", compared, "of them thrown:", thrown_cases, "left out without validation:", skipped_unvalidated)
assert compared > 0

## Bodies that are no request

A body that is not JSON, that is no object, that holds no string `query` or whose `operationName`
is no string answers one error and no data. A `variables` or an `operationName` of `null` reads as
an absent one, as a GraphQL server over HTTP reads it.

In [ ]:
table = json.loads((CORPUS / "blog-app-pylon.json").read_text(encoding="utf-8"))["tables"][0]
name = table["name"]
query = next(c["document"] for c in json.loads((CORPUS / "blog-app-pylon.json").read_text(encoding="utf-8"))["cases"]
             if c["table"] == name and c.get("variables") is None and c.get("operationName") is None
             and c.get("validate", True))
BODIES = [
    ("not json", "{ broken", None),
    ("an array", "[1]", failure("the request body is not a JSON object")),
    ("a string", json.dumps("query"), failure("the request body is not a JSON object")),
    ("no query", "{}", failure("the request carries no string field query")),
    ("a query that is no string", json.dumps({"query": 3}), failure("the request carries no string field query")),
    ("an operationName that is no string", json.dumps({"query": query, "operationName": 3}),
     failure("the field operationName is no string")),
]
requests = [{"id": "t", "op": "table", "table": table}]
for index, (_, body, _) in enumerate(BODIES):
    requests.append({"id": "b%d" % index, "op": "embed", "table": name, "body": body})
plain = {"id": "n0", "op": "embed", "table": name, "body": json.dumps({"query": query})}
nulls = {"id": "n1", "op": "embed", "table": name,
         "body": json.dumps({"query": query, "variables": None, "operationName": None})}
requests += [plain, nulls]
answers = batch(requests, "bodies")
assert answers["t"].get("ok") is True, answers["t"]
differing = 0
for index, (label, body, expected) in enumerate(BODIES):
    got = answers["b%d" % index]["body"]
    if expected is None:
        response = json.loads(got)
        ok = list(response) == ["errors"] and len(response["errors"]) == 1 and \
            response["errors"][0]["message"].startswith("the request body is not JSON: ")
    else:
        ok = got == expected
    print("%-36s %s" % (label, got))
    differing += 0 if ok else 1
print("null fields equal absent ones:", answers["n0"]["body"] == answers["n1"]["body"])
differing += 0 if answers["n0"]["body"] == answers["n1"]["body"] else 1
DIFFS["bodies"] = differing

## A table without a service

The entry needs a service, and a table of the mode `plain` builds none, so the devtool refuses an
`embed` against it as a mistake of the harness, as it refuses a table the run does not hold.

In [ ]:
plain_table = json.loads((CORPUS / "blog-app-plain.json").read_text(encoding="utf-8"))["tables"][0]
answers = batch([
    {"id": "t", "op": "table", "table": plain_table},
    {"id": "p", "op": "embed", "table": plain_table["name"], "body": "{}"},
    {"id": "m", "op": "embed", "table": "no-such-table", "body": "{}"},
], "plain")
print(json.dumps(answers["p"]))
print(json.dumps(answers["m"]))
refusals_ok = (
    answers["p"].get("fatal") == "embed: the table %s is of the mode plain" % plain_table["name"]
    and answers["m"].get("fatal") == "embed: the run holds no table named no-such-table"
)
DIFFS["refusals"] = 0 if refusals_ok else 1

## Verdict

In [ ]:
print("differences by section:", DIFFS)
print("cases compared:", compared)
assert sum(DIFFS.values()) == 0, DIFFS
assert compared > 0